In [ ]:
# class ReinforcementLearningDemo:

#     def __init__(self):
#         self.score = 0
#         self.game_over = False

#     def cartpole_example(self):
#         print("목표 : 막대 쓰러뜨리지 않기")

#         while not self.game_over:
#             state = {
#                 'pole_angle': 0.1,
#                 'cart_position': 0.0,
#                 'pole_velocity': 0.02,
#                 'cart_velocity': 0.1
#             }
#             print(state)

#             action = self.choose_action(state)
#             print(f"선택한 행동: {'왼쪽' if action == -1 else '오른쪽'}")

#             reward = self.calculate_reward(state, action)
#             print(f"받은 보상: {reward}")

#             self.update_policy(state, action, reward)
#             print("전략 업데이트")

#             self.score += reward
#             if self.score < -100:
#                 self.game_over = True

#         print(f"죄종스코어: {self.score}")

#     def choose_action(self, state):
#         import random

#         if random.random() < 0.9:
#             return 1 if state['pole_angle'] > 0 else -1
#         else:
#             return random.choice([-1, 1])

#     def calculate_reward(self, state, action):
#         if abs(state['pole_angle']) > 0.5:
#             return -100
#         else:
#             return +1

#     def update_policy(self, state, action, reward):
#         learning_rate = 0.1
#         discount_factor = 0.95

#         print("더 나은 전략으로 업데이트")

# demo = ReinforcementLearningDemo()
# demo.cartpole_example()

In [1]:
import gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

class QNetwork(nn.Module):
    def __init__(self, state_size, action_size, seed, fc1_units=64, fc2_units=64):
        super(QNetwork, self).__init__()
        self.seed = torch.manual_seed(seed)
        self.fc1 = nn.Linear(state_size, fc1_units)
        self.fc2 = nn.Linear(fc1_units, fc2_units)
        self.fc3 = nn.Linear(fc2_units, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
seed = 1234
qnetwork_local = QNetwork(state_size, action_size, seed)
qnetwork_target = QNetwork(state_size, action_size, seed)
optimizer = optim.Adam(qnetwork_local.parameters(), lr=5e-4)

buffer_size = int(1e5)
batch_size = 64
memory = deque(maxlen=buffer_size)

def step(state, action, reward, next_state, done):
    memory.append((state, action, reward, next_state, done))

def sample():
    experiences = random.sample(memory, k=batch_size)
    states = torch.from_numpy(np.vstack([e[0] for e in experiences])).float()
    actions = torch.from_numpy(np.vstack([e[1] for e in experiences])).long()
    rewards = torch.from_numpy(np.vstack([e[2] for e in experiences])).float()
    next_states = torch.from_numpy(np.vstack([e[3] for e in experiences])).float()
    dones = torch.from_numpy(np.vstack([e[4] for e in experiences]).astype(np.uint8)).float()
    return (states, actions, rewards, next_states, dones)

def learn(experiences, gamma):
    states, actions, rewards, next_states, dones = experiences
    Q_targets_next = qnetwork_target(next_states).detach().max(1)[0].unsqueeze(1)
    Q_targets = rewards + (gamma * Q_targets_next * (1 - dones))
    Q_expected = qnetwork_local(states).gather(1, actions)
    loss = nn.MSELoss()(Q_expected, Q_targets)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

gamma = 0.99
tau = 1e-3
n_episodes = 200
max_t = 1000

for i_episode in range(1, n_episodes+1):
    state = env.reset()
    total_reward = 0
    for t in range(max_t):
        state_tensor = torch.from_numpy(state).float().unsqueeze(0)
        with torch.no_grad():
            action_values = qnetwork_local(state_tensor)
        action = np.argmax(action_values.cpu().data.numpy())
        next_state, reward, done, _ = env.step(action)
        step(state, action, reward, next_state, done)
        total_reward += reward
        if len(memory) > batch_size:
            experiences = sample()
            loss = learn(experiences, gamma)
        state = next_state
        if done:
            break
    for target_param, local_param in zip(qnetwork_target.parameters(), qnetwork_local.parameters()):
        target_param.data.copy_(tau*local_param.data + (1-tau)*target_param.data)
    print(i_episode, total_reward)

env.close()

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
c:\Users\KDS23\Documents\17-deep-learning\.venv\lib\site-packages\gym\core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\KDS23\Documents\17-deep-learning\.venv\lib\site-packages\gym\wrappers\step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future

1 12.0
2 9.0
3 10.0
4 10.0
5 10.0
6 12.0
7 10.0
8 9.0
9 8.0
10 10.0
11 10.0
12 8.0
13 8.0
14 9.0
15 9.0
16 11.0
17 9.0
18 9.0
19 9.0
20 11.0
21 9.0
22 9.0
23 9.0
24 8.0
25 9.0
26 10.0
27 9.0
28 10.0
29 11.0
30 10.0
31 11.0
32 9.0
33 11.0
34 9.0
35 9.0
36 9.0
37 9.0
38 9.0
39 9.0
40 10.0
41 9.0
42 8.0
43 9.0
44 8.0
45 9.0
46 9.0
47 8.0
48 10.0
49 10.0
50 9.0
51 8.0
52 10.0
53 9.0
54 9.0
55 10.0
56 10.0
57 10.0
58 9.0
59 9.0
60 10.0
61 10.0
62 8.0
63 8.0
64 10.0
65 10.0
66 9.0
67 9.0
68 8.0
69 10.0
70 9.0
71 9.0
72 9.0
73 9.0
74 10.0
75 10.0
76 10.0
77 9.0
78 8.0
79 9.0
80 10.0
81 10.0
82 9.0
83 10.0
84 10.0
85 9.0
86 10.0
87 9.0
88 10.0
89 10.0
90 8.0
91 10.0
92 8.0
93 8.0
94 8.0
95 10.0
96 10.0
97 10.0
98 10.0
99 10.0
100 8.0
101 10.0
102 10.0
103 10.0
104 8.0
105 9.0
106 9.0
107 10.0
108 9.0
109 9.0
110 10.0
111 8.0
112 9.0
113 9.0
114 9.0
115 9.0
116 11.0
117 9.0
118 8.0
119 9.0
120 10.0
121 10.0
122 10.0
123 9.0
124 9.0
125 10.0
126 9.0
127 10.0
128 11.0
129 9.0
130 9.0
131 10.0
132

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import gym

class PolicyNetwork(nn.Module):
    def __init__(self, state_size, action_size, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_size, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, action_size)

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        x = self.fc2(x)
        return torch.softmax(x, dim=-1)

env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

policy_net = PolicyNetwork(state_size, action_size)
optimizer = optim.Adam(policy_net.parameters(), lr=0.01)

def select_action(state):
    state = torch.from_numpy(state).float().unsqueeze(0)
    probs = policy_net(state)
    action = torch.multinomial(probs, num_samples=1)
    return action.item(), torch.log(probs[0, action.item()])

def reinforce_update(episode_rewards, episode_log_probs, gamma=0.99):
    R = 0
    returns = []
    for r in episode_rewards[::-1]:
        R = r + gamma * R
        returns.insert(0, R)
    returns = torch.tensor(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    loss = 0
    for log_prob, R in zip(episode_log_probs, returns):
        loss -= log_prob * R
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

num_episodes = 1000
for episode in range(num_episodes):
    state = env.reset()
    episode_rewards = []
    episode_log_probs = []
    done = False
    while not done:
        action, log_prob = select_action(state)
        next_state, reward, done, _ = env.step(action)
        episode_rewards.append(reward)
        episode_log_probs.append(log_prob)
        state = next_state
    loss = reinforce_update(episode_rewards, episode_log_probs)
    if episode % 50 == 0:
        total_reward = sum(episode_rewards)
        print(episode, total_reward, loss)

env.close()

c:\Users\KDS23\Documents\17-deep-learning\.venv\lib\site-packages\gym\core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\KDS23\Documents\17-deep-learning\.venv\lib\site-packages\gym\wrappers\step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\KDS23\Documents\17-deep-learning\.venv\lib\site-packages\gym\utils\passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


0 37.0 -0.06466531753540039
50 23.0 0.21240416169166565
100 109.0 -0.00991201400756836
150 500.0 -8.608781814575195
200 500.0 -9.479695320129395
250 254.0 -6.648103713989258
300 500.0 15.367204666137695
350 104.0 -0.21770347654819489
400 27.0 -1.1643176078796387
450 113.0 -1.7363556623458862
500 446.0 -12.959242820739746
550 500.0 -7.266448020935059
600 500.0 -0.5770056843757629
650 186.0 1.1575151681900024
700 500.0 6.217838287353516
750 30.0 -4.350668430328369
800 158.0 -5.317481994628906
850 21.0 3.587949514389038
900 43.0 -0.132741317152977
950 48.0 -1.321702241897583
